# Code Critic — Watspeed Agentic AI Capstone
**Author**: Jayaprakash Sivanandam &nbsp;|&nbsp; **Date**: 2026-04-26

Agentic SQL code reviewer for Snowflake. Given a stored procedure name or a raw SQL query, an A2A agent graph analyzes performance, security, and style and returns a ranked `FindingsReport`.

**Setup**: add `GOOGLE_API_KEY` and `ANTHROPIC_API_KEY` to `.env`, then *Restart & Run All*.

**Architecture**: Each node runs as a standalone A2A agent service. The orchestrator (Claude Opus) drives the full pipeline — routing, schema fetching, parallel analysis, and synthesis. Analyzer agents use Gemma for speed.

| Step | Port | Node | Model | Description |
|------|------|------|-------|-------------|
| 1 | 8011 | `router` | Gemma | Classify input as `sql_query`, `stored_procedure`, or `unknown` |
| 2 | 8012 | `schema-fetcher` | Gemma | Fetch Snowflake DDL and extract executable ETL code body |
| 3 | 8013 | `perf-analyzer` | Gemma | Detect performance inefficiencies (parallel) |
| 4 | 8014 | `security-auditor` | Gemma | Flag security vulnerabilities (parallel) |
| 5 | 8015 | `style-reviewer` | Gemma | Catch style and readability violations (parallel) |
| 6 | 8016 | `orchestrator` | Opus | Drive full pipeline; synthesize `FindingsReport` |

In [ ]:
import sys

# installs into the active .venv kernel — safe to re-run
# !{sys.executable} -m pip install -q "a2a-sdk" "protobuf>=5.28.0,<6" uvicorn httpx

In [ ]:
import sys
import pathlib
import asyncio

# Make project root importable so prompts.py can be found from notebooks/
sys.path.insert(0, str(pathlib.Path("..").resolve()))

# ── model definitions ─────────────────────────────────────────────────────────
GEMMA_MODEL  = "gemma-4-26b-a4b-it"
OPUS_MODEL   = "claude-opus-4-6"
SONNET_MODEL = "claude-sonnet-4-6"
# ─────────────────────────────────────────────────────────────────────────────

import os
import json
from typing import Any, Optional
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic

load_dotenv()

gemma_llm  = ChatGoogleGenerativeAI(model=GEMMA_MODEL)
opus_llm   = ChatAnthropic(model=OPUS_MODEL)
sonnet_llm = ChatAnthropic(model=SONNET_MODEL)

print(f"GEMMA  : {GEMMA_MODEL}")
print(f"OPUS   : {OPUS_MODEL}")
print(f"SONNET : {SONNET_MODEL}")


In [ ]:
# MCP client helpers for a separately running FastMCP server
import os
from typing import Any
from fastmcp import Client

# Example: export CODE_CRITIC_MCP_SERVER_URL=http://127.0.0.1:9000/mcp
MCP_SERVER_URL = os.getenv("CODE_CRITIC_MCP_SERVER_URL", "http://127.0.0.1:9000/mcp")

def _extract_text_from_tool_result(tool_result: Any) -> str:
    content = getattr(tool_result, "content", None)
    if content:
        chunks = []
        for item in content:
            text = getattr(item, "text", None)
            if text is not None:
                chunks.append(str(text))
            elif isinstance(item, dict) and "text" in item:
                chunks.append(str(item["text"]))
        if chunks:
            return "\n".join(chunks).strip()

    if isinstance(tool_result, str):
        return tool_result.strip()

    return str(tool_result).strip()

def _json_safe(value: Any) -> Any:
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    if hasattr(value, "model_dump"):
        return _json_safe(value.model_dump())
    if hasattr(value, "__dict__"):
        return _json_safe(value.__dict__)
    return str(value)

async def list_mcp_tools() -> list[dict[str, Any]]:
    async with Client(MCP_SERVER_URL) as client:
        tools = await client.list_tools()

    normalized = []
    for tool in tools:
        normalized.append(
            {
                "name": getattr(tool, "name", ""),
                "description": getattr(tool, "description", ""),
                "input_schema": _json_safe(
                    getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None)
                ),
            }
        )
    return normalized

async def call_mcp_tool(tool_name: str, arguments: dict[str, Any] | None = None) -> str:
    async with Client(MCP_SERVER_URL) as client:
        tool_result = await client.call_tool(tool_name, arguments or {})
    return _extract_text_from_tool_result(tool_result)

async def get_mcp_prompt(name: str) -> str:
    """Fetch a named prompt from the running MCP server and return its text content."""
    async with Client(MCP_SERVER_URL) as client:
        result = await client.get_prompt(name)
    for msg in result.messages:
        content = getattr(msg, "content", None)
        if content is not None:
            text = getattr(content, "text", None)
            if text:
                return text
    return ""


In [ ]:
# ── Load all skill prompts from the MCP server ────────────────────────────────
# Requires the MCP server to be running: python mcp_server.py
ROUTER_SKILL               = await get_mcp_prompt("router_skill")
SCHEMA_FETCHER_SKILL       = await get_mcp_prompt("schema_fetcher_skill")
SCHEMA_FETCHER_ETL_PROMPT  = await get_mcp_prompt("schema_fetcher_etl_prompt")
PERFORMANCE_ANALYZER_SKILL = await get_mcp_prompt("performance_analyzer_skill")
SECURITY_AUDITOR_SKILL     = await get_mcp_prompt("security_auditor_skill")
STYLE_REVIEWER_SKILL       = await get_mcp_prompt("style_reviewer_skill")
SYNTHESIZER_SKILL          = await get_mcp_prompt("synthesizer_skill")

print("Prompts loaded from MCP server:")
for name in ["ROUTER_SKILL", "SCHEMA_FETCHER_SKILL", "SCHEMA_FETCHER_ETL_PROMPT",
             "PERFORMANCE_ANALYZER_SKILL", "SECURITY_AUDITOR_SKILL",
             "STYLE_REVIEWER_SKILL", "SYNTHESIZER_SKILL"]:
    val = locals()[name]
    print(f"  {name}: {len(val)} chars")


In [ ]:
from enum import Enum

class InputType(str, Enum):
    SQL_QUERY        = "sql_query"
    STORED_PROCEDURE = "stored_procedure"
    UNKNOWN          = "unknown"

class RouterDecision(BaseModel):
    input_type:     InputType
    confidence:     float
    reasoning:      str
    object_name:    Optional[str] = None
    original_input: str = ""


In [ ]:
import threading
import time
import uvicorn
from uuid import uuid4

from fastapi import FastAPI
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes.jsonrpc_routes import create_jsonrpc_routes
from a2a.server.routes.agent_card_routes import create_agent_card_routes
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater
from a2a.types import AgentCard, AgentCapabilities, AgentSkill, a2a_pb2
from a2a.utils import TransportProtocol

ROUTER_PORT = 8011

class RouterAgentExecutor(AgentExecutor):
    def __init__(self):
        self._chain = gemma_llm.with_structured_output(RouterDecision)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )

        user_input = context.get_user_input()
        decision = self._chain.invoke([SystemMessage(ROUTER_SKILL), HumanMessage(user_input)])

        # Always preserve the exact original input for downstream consumers.
        decision.original_input = user_input

        # Normalize procedure object names so downstream GET_DDL calls can resolve signatures.
        if decision.input_type == InputType.STORED_PROCEDURE and decision.object_name:
            normalized_object_name = decision.object_name.strip()
            if "(" not in normalized_object_name:
                normalized_object_name = f"{normalized_object_name}()"
            decision.object_name = normalized_object_name
            
        print(f"Router decision: {decision.model_dump_json()}")
        
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(
            parts=[a2a_pb2.Part(text=decision.model_dump_json())]
        )
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Router agent does not support cancellation")


router_card = AgentCard(
    name="router",
    description="Classify raw user input as sql_query, stored_procedure, or unknown.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{ROUTER_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="route",
            name="Route Input",
            description="Classify SQL input type and extract object name if applicable.",
            tags=["routing", "sql"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

router_handler = DefaultRequestHandler(
    agent_executor=RouterAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=router_card,
)

router_routes = create_agent_card_routes(agent_card=router_card) + create_jsonrpc_routes(
    request_handler=router_handler,
    rpc_url="/",
)
router_app = FastAPI(
    title="Router Agent",
    description="Classify SQL input type and route to the appropriate pipeline.",
    version="1.0.0",
)
router_app.routes.extend(router_routes)


In [ ]:
class SchemaFetchResult(BaseModel):
    object_name: str
    object_type: str
    sql_text: str
    source: str = "snowflake_ddl"


class SchemaFetcherAgentExecutor(AgentExecutor):
    @staticmethod
    def _normalize_object_name(name: str) -> str:
        """Ensure a stored procedure name always ends with () for GET_DDL compatibility."""
        name = name.strip()
        if "(" not in name:
            name = f"{name}()"
        print(f"Normalized object name: {name}")
        return name

    @staticmethod
    def _extract_ddl_from_tool_output(tool_output: str) -> str:
        """Return DDL text from schema fetcher MCP tool output."""
        text = (tool_output or "").strip()
        if not text:
            return ""

        # If tool returned plain DDL text, use it directly.
        if not (text.startswith("{") or text.startswith("[")):
            return text

        payload = None
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            try:
                import ast

                payload = ast.literal_eval(text)
            except Exception:
                return text

        if isinstance(payload, dict):
            if payload.get("status") == "error":
                raise RuntimeError(payload.get("message") or payload.get("fetch_error") or "Schema fetcher tool failed")
            for key in ("ddl", "raw_ddl", "sql_text", "etl_code"):
                value = payload.get(key)
                if isinstance(value, str) and value.strip():
                    return value.strip()

        return text

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )

        object_name = self._normalize_object_name(context.get_user_input())

        tool_output = await call_mcp_tool(
            "schema_fetcher_tool",
            {
                "object_name": object_name,
                # "object_type": "PROCEDURE",
            },
        )
        ddl_text = self._extract_ddl_from_tool_output(tool_output)

        if not ddl_text:
            raise RuntimeError("Schema fetcher tool did not return DDL text")

        result = SchemaFetchResult(
            object_name=object_name.upper(),
            object_type="PROCEDURE",
            sql_text=ddl_text,
        )

        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(
            parts=[a2a_pb2.Part(text=result.model_dump_json())]
        )
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Schema fetcher does not support cancellation")


SCHEMA_FETCHER_PORT = 8012

schema_fetcher_card = AgentCard(
    name="schema-fetcher",
    description="Fetch Snowflake object DDL and return it directly.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{SCHEMA_FETCHER_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="fetch_schema",
            name="Fetch SQL Code",
            description="Fetch procedure definition and return raw DDL.",
            tags=["schema", "snowflake", "ddl"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

schema_fetcher_handler = DefaultRequestHandler(
    agent_executor=SchemaFetcherAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=schema_fetcher_card,
)

schema_fetcher_routes = create_agent_card_routes(
    agent_card=schema_fetcher_card,
) + create_jsonrpc_routes(
    request_handler=schema_fetcher_handler,
    rpc_url="/",
)

schema_fetcher_app = FastAPI(
    title="Schema Fetcher Agent",
    description="Fetches Snowflake DDL and returns it directly.",
    version="1.0.0",
)
schema_fetcher_app.routes.extend(schema_fetcher_routes)

In [ ]:
# ── Analyzer data models ──────────────────────────────────────────────────────

class Severity(str, Enum):
    HIGH   = "high"
    MEDIUM = "medium"
    LOW    = "low"
    INFO   = "info"

class Finding(BaseModel):
    severity:    Severity
    description: str
    snippet:     Optional[str] = None
    suggestion:  str

class AnalysisResult(BaseModel):
    findings: list[Finding]

class FindingsReport(BaseModel):
    input_type:           str
    object_name:          Optional[str] = None
    performance_findings: list[Finding]
    security_findings:    list[Finding]
    style_findings:       list[Finding]
    rewritten_sql:        Optional[str] = None
    summary:              str

In [ ]:
PERF_ANALYZER_PORT = 8013

class PerformanceAnalyzerAgentExecutor(AgentExecutor):
    def __init__(self):
        self._chain = gemma_llm.with_structured_output(AnalysisResult)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )
        sql_text = context.get_user_input()
        result = self._chain.invoke([SystemMessage(PERFORMANCE_ANALYZER_SKILL), HumanMessage(sql_text)])

        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(parts=[a2a_pb2.Part(text=result.model_dump_json())])
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Performance analyzer does not support cancellation")


perf_analyzer_card = AgentCard(
    name="performance-analyzer",
    description="Detect performance inefficiencies in SQL queries and stored procedures.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{PERF_ANALYZER_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="analyze_performance",
            name="Analyze Performance",
            description="Find performance issues in SQL code.",
            tags=["performance", "sql"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

perf_analyzer_handler = DefaultRequestHandler(
    agent_executor=PerformanceAnalyzerAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=perf_analyzer_card,
)

perf_analyzer_routes = create_agent_card_routes(agent_card=perf_analyzer_card) + create_jsonrpc_routes(
    request_handler=perf_analyzer_handler,
    rpc_url="/",
)
perf_analyzer_app = FastAPI(title="Performance Analyzer Agent", version="1.0.0")
perf_analyzer_app.routes.extend(perf_analyzer_routes)

In [ ]:
SECURITY_AUDITOR_PORT = 8014

class SecurityAuditorAgentExecutor(AgentExecutor):
    def __init__(self):
        self._chain = gemma_llm.with_structured_output(AnalysisResult)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )
        sql_text = context.get_user_input()
        result = self._chain.invoke([SystemMessage(SECURITY_AUDITOR_SKILL), HumanMessage(sql_text)])

        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(parts=[a2a_pb2.Part(text=result.model_dump_json())])
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Security auditor does not support cancellation")


security_auditor_card = AgentCard(
    name="security-auditor",
    description="Flag security vulnerabilities in SQL queries and stored procedures.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{SECURITY_AUDITOR_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="audit_security",
            name="Audit Security",
            description="Flag security vulnerabilities in SQL code.",
            tags=["security", "sql"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

security_auditor_handler = DefaultRequestHandler(
    agent_executor=SecurityAuditorAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=security_auditor_card,
)

security_auditor_routes = create_agent_card_routes(agent_card=security_auditor_card) + create_jsonrpc_routes(
    request_handler=security_auditor_handler,
    rpc_url="/",
)
security_auditor_app = FastAPI(title="Security Auditor Agent", version="1.0.0")
security_auditor_app.routes.extend(security_auditor_routes)

In [ ]:
STYLE_REVIEWER_PORT = 8015

class StyleReviewerAgentExecutor(AgentExecutor):
    def __init__(self):
        self._chain = gemma_llm.with_structured_output(AnalysisResult)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )
        sql_text = context.get_user_input()
        result = self._chain.invoke([SystemMessage(STYLE_REVIEWER_SKILL), HumanMessage(sql_text)])

        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(parts=[a2a_pb2.Part(text=result.model_dump_json())])
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Style reviewer does not support cancellation")


style_reviewer_card = AgentCard(
    name="style-reviewer",
    description="Catch style and readability violations in SQL queries and stored procedures.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{STYLE_REVIEWER_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="review_style",
            name="Review Style",
            description="Flag style and readability violations in SQL code.",
            tags=["style", "sql"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

style_reviewer_handler = DefaultRequestHandler(
    agent_executor=StyleReviewerAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=style_reviewer_card,
)

style_reviewer_routes = create_agent_card_routes(agent_card=style_reviewer_card) + create_jsonrpc_routes(
    request_handler=style_reviewer_handler,
    rpc_url="/",
)
style_reviewer_app = FastAPI(title="Style Reviewer Agent", version="1.0.0")
style_reviewer_app.routes.extend(style_reviewer_routes)

In [ ]:
from a2a.client import (
    A2AClientTimeoutError,
    ClientCallContext,
    ClientFactory,
)

ROUTER_TIMEOUT_SECONDS          = 300.0
SCHEMA_FETCHER_TIMEOUT_SECONDS  = 300.0
ANALYZER_TIMEOUT_SECONDS        = 300.0
ORCHESTRATOR_TIMEOUT_SECONDS    = 1500.0


async def call_router(user_input: str) -> RouterDecision:
    factory = ClientFactory()
    client = await factory.create_from_url(f"http://localhost:{ROUTER_PORT}")

    request = a2a_pb2.SendMessageRequest()
    request.message.role = a2a_pb2.ROLE_USER
    request.message.message_id = str(uuid4())
    request.message.parts.add().text = user_input

    call_context = ClientCallContext(timeout=ROUTER_TIMEOUT_SECONDS)
    try:
        async for event in client.send_message(request, context=call_context):
            if event.HasField("status_update"):
                status = event.status_update.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return RouterDecision.model_validate_json(status.message.parts[0].text)
            if event.HasField("task"):
                status = event.task.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return RouterDecision.model_validate_json(status.message.parts[0].text)
            if event.HasField("message") and event.message.parts:
                return RouterDecision.model_validate_json(event.message.parts[0].text)
    except A2AClientTimeoutError as exc:
        raise RuntimeError(
            f"Router call timed out after {ROUTER_TIMEOUT_SECONDS:.0f}s. "
            "Ensure the router server is running and API keys are set."
        ) from exc
    raise RuntimeError("No completed response received from router")


async def call_schema_fetcher(object_name: str) -> SchemaFetchResult:
    factory = ClientFactory()
    client = await factory.create_from_url(f"http://localhost:{SCHEMA_FETCHER_PORT}")

    request = a2a_pb2.SendMessageRequest()
    request.message.role = a2a_pb2.ROLE_USER
    request.message.message_id = str(uuid4())
    request.message.parts.add().text = object_name

    call_context = ClientCallContext(timeout=SCHEMA_FETCHER_TIMEOUT_SECONDS)
    try:
        async for event in client.send_message(request, context=call_context):
            if event.HasField("status_update") and event.status_update.status.message.parts:
                status = event.status_update.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED:
                    return SchemaFetchResult.model_validate_json(status.message.parts[0].text)
            if event.HasField("task") and event.task.status.message.parts:
                status = event.task.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED:
                    return SchemaFetchResult.model_validate_json(status.message.parts[0].text)
            if event.HasField("message") and event.message.parts:
                return SchemaFetchResult.model_validate_json(event.message.parts[0].text)
    except A2AClientTimeoutError as exc:
        raise RuntimeError(
            f"Schema fetcher timed out after {SCHEMA_FETCHER_TIMEOUT_SECONDS:.0f}s."
        ) from exc
    raise RuntimeError("No completed response received from schema fetcher")


async def _call_analyzer(port: int, sql_text: str) -> AnalysisResult:
    factory = ClientFactory()
    client = await factory.create_from_url(f"http://localhost:{port}")

    request = a2a_pb2.SendMessageRequest()
    request.message.role = a2a_pb2.ROLE_USER
    request.message.message_id = str(uuid4())
    request.message.parts.add().text = sql_text

    call_context = ClientCallContext(timeout=ANALYZER_TIMEOUT_SECONDS)
    try:
        async for event in client.send_message(request, context=call_context):
            if event.HasField("status_update"):
                status = event.status_update.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return AnalysisResult.model_validate_json(status.message.parts[0].text)
            if event.HasField("task"):
                status = event.task.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return AnalysisResult.model_validate_json(status.message.parts[0].text)
            if event.HasField("message") and event.message.parts:
                return AnalysisResult.model_validate_json(event.message.parts[0].text)
    except A2AClientTimeoutError as exc:
        raise RuntimeError(
            f"Analyzer on port {port} timed out after {ANALYZER_TIMEOUT_SECONDS:.0f}s."
        ) from exc
    raise RuntimeError(f"No completed response received from analyzer on port {port}")


async def call_perf_analyzer(sql_text: str) -> AnalysisResult:
    return await _call_analyzer(PERF_ANALYZER_PORT, sql_text)

async def call_security_auditor(sql_text: str) -> AnalysisResult:
    return await _call_analyzer(SECURITY_AUDITOR_PORT, sql_text)

async def call_style_reviewer(sql_text: str) -> AnalysisResult:
    return await _call_analyzer(STYLE_REVIEWER_PORT, sql_text)


async def call_orchestrator(user_input: str) -> FindingsReport:
    factory = ClientFactory()
    client = await factory.create_from_url(f"http://localhost:{ORCHESTRATOR_PORT}")

    request = a2a_pb2.SendMessageRequest()
    request.message.role = a2a_pb2.ROLE_USER
    request.message.message_id = str(uuid4())
    request.message.parts.add().text = user_input

    call_context = ClientCallContext(timeout=ORCHESTRATOR_TIMEOUT_SECONDS)
    try:
        async for event in client.send_message(request, context=call_context):
            if event.HasField("status_update"):
                status = event.status_update.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return FindingsReport.model_validate_json(status.message.parts[0].text)
            if event.HasField("task"):
                status = event.task.status
                if status.state == a2a_pb2.TASK_STATE_COMPLETED and status.message.parts:
                    return FindingsReport.model_validate_json(status.message.parts[0].text)
            if event.HasField("message") and event.message.parts:
                return FindingsReport.model_validate_json(event.message.parts[0].text)
    except A2AClientTimeoutError as exc:
        raise RuntimeError(
            f"Orchestrator timed out after {ORCHESTRATOR_TIMEOUT_SECONDS:.0f}s."
        ) from exc
    raise RuntimeError("No completed response received from orchestrator")


In [ ]:
ORCHESTRATOR_PORT = 8016

class SynthesisOutput(BaseModel):
    summary: str
    rewritten_sql: Optional[str] = None

class OrchestratorAgentExecutor(AgentExecutor):
    def __init__(self):
        self._synthesis_chain = opus_llm.with_structured_output(SynthesisOutput)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        await event_queue.enqueue_event(
            a2a_pb2.Task(
                id=context.task_id,
                context_id=context.context_id,
                status=a2a_pb2.TaskStatus(state=a2a_pb2.TASK_STATE_WORKING),
            )
        )

        user_input = context.get_user_input()
        decision = await call_router(user_input)
        
        print(f"Orchestrator received router decision: {decision.model_dump_json()}")

        if decision.input_type == InputType.STORED_PROCEDURE and decision.object_name:
            # Prefer original_input (exact user text) over the LLM-extracted object_name.
            fetch_target = decision.original_input.strip() or decision.object_name
            schema_result = await call_schema_fetcher(fetch_target)
            sql_text = schema_result.sql_text
        elif decision.input_type == InputType.SQL_QUERY:
            sql_text = user_input
        else:
            report = FindingsReport(
                input_type=decision.input_type.value,
                object_name=decision.object_name,
                performance_findings=[],
                security_findings=[],
                style_findings=[],
                summary="Input could not be classified as SQL or a stored procedure.",
            )
            updater = TaskUpdater(event_queue, context.task_id, context.context_id)
            reply = updater.new_agent_message(parts=[a2a_pb2.Part(text=report.model_dump_json())])
            await updater.complete(message=reply)
            return

        perf_result, security_result, style_result = await asyncio.gather(
            call_perf_analyzer(sql_text),
            call_security_auditor(sql_text),
            call_style_reviewer(sql_text),
        )

        synthesis_prompt = (
            f"input_type: {decision.input_type.value}\n"
            f"object_name: {decision.object_name or 'N/A'}\n\n"
            f"SQL:\n{sql_text}\n\n"
            f"Performance findings:\n{perf_result.model_dump_json()}\n\n"
            f"Security findings:\n{security_result.model_dump_json()}\n\n"
            f"Style findings:\n{style_result.model_dump_json()}"
        )
        synthesis = self._synthesis_chain.invoke([
            SystemMessage(SYNTHESIZER_SKILL),
            HumanMessage(synthesis_prompt),
        ])

        report = FindingsReport(
            input_type=decision.input_type.value,
            object_name=decision.object_name,
            performance_findings=perf_result.findings,
            security_findings=security_result.findings,
            style_findings=style_result.findings,
            rewritten_sql=synthesis.rewritten_sql,
            summary=synthesis.summary,
        )
        updater = TaskUpdater(event_queue, context.task_id, context.context_id)
        reply = updater.new_agent_message(parts=[a2a_pb2.Part(text=report.model_dump_json())])
        await updater.complete(message=reply)

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        raise NotImplementedError("Orchestrator does not support cancellation")


orchestrator_card = AgentCard(
    name="orchestrator",
    description="Drive the full SQL code review pipeline: route → fetch → analyze → synthesize.",
    supported_interfaces=[
        a2a_pb2.AgentInterface(
            url=f"http://localhost:{ORCHESTRATOR_PORT}",
            protocol_binding=TransportProtocol.JSONRPC,
            protocol_version="1.0",
        )
    ],
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="orchestrate",
            name="Orchestrate Review",
            description="Full pipeline: classify input, fetch schema if needed, run all analyzers, synthesize report.",
            tags=["orchestration", "sql", "review"],
            input_modes=["text"],
            output_modes=["text"],
        )
    ],
)

orchestrator_handler = DefaultRequestHandler(
    agent_executor=OrchestratorAgentExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=orchestrator_card,
)

orchestrator_routes = create_agent_card_routes(agent_card=orchestrator_card) + create_jsonrpc_routes(
    request_handler=orchestrator_handler,
    rpc_url="/",
)
orchestrator_app = FastAPI(title="Orchestrator Agent", version="1.0.0")
orchestrator_app.routes.extend(orchestrator_routes)


In [ ]:
# ── Launch all A2A agent servers ──────────────────────────────────────────────

router_server = schema_fetcher_server = perf_analyzer_server = None
security_auditor_server = style_reviewer_server = orchestrator_server = None


def _serve_router():
    global router_server
    config = uvicorn.Config(router_app, host="localhost", port=ROUTER_PORT, log_level="warning")
    router_server = uvicorn.Server(config)
    router_server.run()


def _serve_schema_fetcher():
    global schema_fetcher_server
    config = uvicorn.Config(schema_fetcher_app, host="localhost", port=SCHEMA_FETCHER_PORT, log_level="warning")
    schema_fetcher_server = uvicorn.Server(config)
    schema_fetcher_server.run()


def _serve_perf_analyzer():
    global perf_analyzer_server
    config = uvicorn.Config(perf_analyzer_app, host="localhost", port=PERF_ANALYZER_PORT, log_level="warning")
    perf_analyzer_server = uvicorn.Server(config)
    perf_analyzer_server.run()


def _serve_security_auditor():
    global security_auditor_server
    config = uvicorn.Config(security_auditor_app, host="localhost", port=SECURITY_AUDITOR_PORT, log_level="warning")
    security_auditor_server = uvicorn.Server(config)
    security_auditor_server.run()


def _serve_style_reviewer():
    global style_reviewer_server
    config = uvicorn.Config(style_reviewer_app, host="localhost", port=STYLE_REVIEWER_PORT, log_level="warning")
    style_reviewer_server = uvicorn.Server(config)
    style_reviewer_server.run()


def _serve_orchestrator():
    global orchestrator_server
    config = uvicorn.Config(orchestrator_app, host="localhost", port=ORCHESTRATOR_PORT, log_level="warning")
    orchestrator_server = uvicorn.Server(config)
    orchestrator_server.run()


router_thread           = threading.Thread(target=_serve_router,           daemon=True)
schema_fetcher_thread   = threading.Thread(target=_serve_schema_fetcher,   daemon=True)
perf_analyzer_thread    = threading.Thread(target=_serve_perf_analyzer,    daemon=True)
security_auditor_thread = threading.Thread(target=_serve_security_auditor, daemon=True)
style_reviewer_thread   = threading.Thread(target=_serve_style_reviewer,   daemon=True)
orchestrator_thread     = threading.Thread(target=_serve_orchestrator,     daemon=True)

for t in [router_thread, schema_fetcher_thread, perf_analyzer_thread,
          security_auditor_thread, style_reviewer_thread, orchestrator_thread]:
    t.start()

time.sleep(3)
print("All agents running:")
print(f"  router           → http://localhost:{ROUTER_PORT}")
print(f"  schema-fetcher   → http://localhost:{SCHEMA_FETCHER_PORT}")
print(f"  perf-analyzer    → http://localhost:{PERF_ANALYZER_PORT}")
print(f"  security-auditor → http://localhost:{SECURITY_AUDITOR_PORT}")
print(f"  style-reviewer   → http://localhost:{STYLE_REVIEWER_PORT}")
print(f"  orchestrator     → http://localhost:{ORCHESTRATOR_PORT}")

In [ ]:
# ── Set the input and run the full pipeline ───────────────────────────────────
# user_input = "BRONZE.PUBLIC.BUILD_END_TABLE_TRICKY_FIXED_V3()"
user_input = "SELECT order_id, SUM(amount) FROM orders WHERE status = 'open' GROUP BY 1"

report = await call_orchestrator(user_input)

print(f"input_type : {report.input_type}")
if report.object_name:
    print(f"object     : {report.object_name}")
print(f"\nsummary:\n{report.summary}")
print(f"\nperformance findings : {len(report.performance_findings)}")
print(f"security findings    : {len(report.security_findings)}")
print(f"style findings       : {len(report.style_findings)}")
if report.rewritten_sql:
    print(f"\nrewritten SQL preview:\n{report.rewritten_sql[:500]}")
    
    
    

In [ ]:
# Stop all agent servers started earlier
_servers = [
    ("router",             globals().get("router_server"),             globals().get("router_thread")),
    ("schema-fetcher",     globals().get("schema_fetcher_server"),     globals().get("schema_fetcher_thread")),
    ("perf-analyzer",      globals().get("perf_analyzer_server"),      globals().get("perf_analyzer_thread")),
    ("security-auditor",   globals().get("security_auditor_server"),   globals().get("security_auditor_thread")),
    ("style-reviewer",     globals().get("style_reviewer_server"),     globals().get("style_reviewer_thread")),
    ("orchestrator",       globals().get("orchestrator_server"),       globals().get("orchestrator_thread")),
]

for name, server, thread in _servers:
    if server is not None:
        server.should_exit = True
        if thread is not None and thread.is_alive():
            thread.join(timeout=3)
        print(f"{name} stopped.")
    else:
        print(f"{name} was not running.")